## Récupération des données

In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Model
from sklearn.ensemble import RandomForestRegressor

# Option pour afficher toutes les colonnes
pd.set_option('display.max_columns', None)

## Récupération des données

In [3]:
print("Chargement des données en cours...")
start_time = time.time()

X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

# Pour y, on s'assure d'avoir un vecteur plat (1D)
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

load_time = time.time() - start_time
print(f"Données chargées en {load_time:.2f} secondes.")
print(f"Dimensions Train : {X_train.shape} | Dimensions Test : {X_test.shape}")

Chargement des données en cours...
Données chargées en 9.15 secondes.
Dimensions Train : (3712444, 13) | Dimensions Test : (928111, 13)


## Stratégie entrainement


In [4]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import time

# 1. Mise à l'échelle (Crucial pour Ridge)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Création du modèle
ridge_model = Ridge(alpha=1.0) # alpha est le paramètre de régularisation

print("Lancement de la Régression Ridge...")
start_time = time.time()
ridge_model.fit(X_train_scaled, y_train)
end_time = time.time() - start_time

print(f"Entraînement terminé en {end_time:.2f} secondes.")

# 3. Prédictions
y_pred_ridge = ridge_model.predict(X_test_scaled)

# 4. Évaluation
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"--- RÉSULTATS RIDGE ---")
print(f"R² Score : {r2_ridge:.4f}")
print(f"MAE      : {mae_ridge:.2f} €")

Lancement de la Régression Ridge...
Entraînement terminé en 0.82 secondes.
--- RÉSULTATS RIDGE ---
R² Score : 0.4523
MAE      : 93662.22 €


### XGBoost (0,75) vs Ridge (0,45) : Il y a un gouffre de 30 points de précision.

En euros (MAE) : XGBoost se trompe de ~59k€, tandis que Ridge se trompe de ~94k€. Ton travail sur les algorithmes complexes permet donc d'économiser en moyenne 35 000 € d'erreur par estimation.

La Régression Ridge est un modèle "rigide" (linéaire). Elle essaie de tracer une ligne droite dans un monde qui ne l'est pas.L'effet de la localisation : Ridge ne comprend pas que le prix peut doubler juste en traversant une rue (effet de quartier/iris). Pour lui, la longitude et la latitude ne sont que des pentes constantes.L'effet de surface : Il ne capte pas que les premiers $m^2$ coûtent plus cher que les derniers (dégressivité).

Les résultats de la Régression Ridge (R² = 0,45) servent de point de référence. Cette performance médiocre met en évidence la nature non-linéaire des données immobilières DVF. L'écart massif de performance (+30% de précision) en faveur de XGBoost justifie pleinement l'investissement en temps de calcul et la complexité des modèles d'ensemble pour répondre à notre problématique d'estimation.